# 🧠 Module 05: Memory & Conversation History

---

## Why Memory Matters

LLMs are **stateless** — each API call is independent. Without memory, your chatbot forgets everything after each message:

```
User: "My name is Alice."
Bot:  "Hi Alice!"
User: "What's my name?"
Bot:  "I don't know your name."  ← No memory!
```

**Memory** solves this by storing and passing conversation history.

---

## Modern Memory Approach (LangChain v0.2+)

LangChain moved away from the old `ConversationBufferMemory` to a more explicit, flexible approach:

| Storage Type | Where History Lives |
|-------------|--------------------|
| **In-memory** | Python list/dict (per session) |
| **File-based** | JSON file (persists across restarts) |
| **Database** | Redis, PostgreSQL, SQLite |
| **Cloud** | Upstash Redis, DynamoDB |

---

In [1]:
import os
from dotenv import load_dotenv
load_dotenv(dotenv_path="../.env")

from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser

llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0.5)
print("Setup complete ✅")

Setup complete ✅


## 1️⃣ Manual Message History (Simplest Approach)

In [2]:
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

# ============================================================
# Maintain a growing list of messages
# ============================================================
conversation_history = [
    SystemMessage(content="You are a friendly, helpful assistant named Nova.")
]

def chat(user_message: str) -> str:
    """Send a message and update conversation history"""
    # Add user message to history
    conversation_history.append(HumanMessage(content=user_message))
    
    # Get response (passing full history)
    response = llm.invoke(conversation_history)
    
    # Add AI response to history
    conversation_history.append(AIMessage(content=response.content))
    
    return response.content

# Simulate a multi-turn conversation
turns = [
    "Hi! My name is Sujat and I'm a Senior data Analyst.",
    "I'm learning LangChain. What should I focus on?",
    "Wait, do you remember what my job is?"
]

for turn in turns:
    print(f"👤 User: {turn}")
    response = chat(turn)
    print(f"🤖 Nova: {response}")
    print()

👤 User: Hi! My name is Sujat and I'm a Senior data Analyst.
🤖 Nova: Hello Sujat, nice to meet you. It's great to connect with a Senior Data Analyst like yourself. What brings you here today? Are you working on a project and need some assistance, or perhaps you'd like to discuss some data analysis topics or trends? I'm all ears and here to help.

👤 User: I'm learning LangChain. What should I focus on?
🤖 Nova: LangChain is a fascinating framework for building applications with large language models. As a Senior Data Analyst, you likely have a strong foundation in data analysis and interpretation, which will serve you well in learning LangChain.

To get started, I'd recommend focusing on the following key areas:

1. **Understanding the LangChain Architecture**: Familiarize yourself with the overall architecture of LangChain, including the different components such as the LLM (Large Language Model), the Index, and the Agent.
2. **LLM Fundamentals**: Learn about the capabilities and limitat

In [3]:
# Inspect the full conversation history
print("Full conversation history:")
print(f"Total messages: {len(conversation_history)}")
print()
for msg in conversation_history:
    role = type(msg).__name__.replace('Message', '')
    print(f"[{role}]: {msg.content[:80]}..." if len(msg.content) > 80 else f"[{role}]: {msg.content}")

Full conversation history:
Total messages: 7

[System]: You are a friendly, helpful assistant named Nova.
[Human]: Hi! My name is Sujat and I'm a Senior data Analyst.
[AI]: Hello Sujat, nice to meet you. It's great to connect with a Senior Data Analyst ...
[Human]: I'm learning LangChain. What should I focus on?
[AI]: LangChain is a fascinating framework for building applications with large langua...
[Human]: Wait, do you remember what my job is?
[AI]: You're a Senior Data Analyst, Sujat. I recall that from our earlier conversation...


## 2️⃣ ChatMessageHistory — Standard Storage

In [4]:
from langchain_core.chat_history import InMemoryChatMessageHistory

# ============================================================
# InMemoryChatMessageHistory — proper history object
# ============================================================
history = InMemoryChatMessageHistory()

# Add messages
history.add_user_message("My favorite color is blue.")
history.add_ai_message("That's lovely! Blue is calming.")
history.add_user_message("What's my favorite color?")

print("Messages in history:")
for msg in history.messages:
    print(f"  {type(msg).__name__}: {msg.content}")

print(f"\nTotal: {len(history.messages)} messages")

Messages in history:
  HumanMessage: My favorite color is blue.
  AIMessage: That's lovely! Blue is calming.
  HumanMessage: What's my favorite color?

Total: 3 messages


## 3️⃣ RunnableWithMessageHistory — Production-Ready Chatbot

In [7]:
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.chat_history import InMemoryChatMessageHistory

# ============================================================
# Build a chatbot with automatic history management
# ============================================================

# Chat prompt with history placeholder
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful coding tutor. Be encouraging and clear."),
    MessagesPlaceholder(variable_name="history"),  # History injected here
    ("human", "{input}")
])

chain = prompt | llm | StrOutputParser()

# Session store — maps session_id to history
session_store = {}

def get_session_history(session_id: str) -> InMemoryChatMessageHistory:
    """Get or create history for a session"""
    if session_id not in session_store:
        session_store[session_id] = InMemoryChatMessageHistory()
    return session_store[session_id]

# Wrap chain with history management
chatbot = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history"
)

# Session config — identifies which conversation
config = {"configurable": {"session_id": "user_alice"}}

# Have a conversation
print("Starting conversation with Alice:")
print("=" * 50)

messages = [
    "Hi! I'm Sujat and I'm learning Python.",
    "What topic should I study after learning variables?",
    "Can you give me a simple example of what you just suggested?",
    "By the way, what was my name again?"
]

for msg in messages:
    print(f"\n👤 Sujat: {msg}")
    response = chatbot.invoke({"input": msg}, config=config)
    print(f"🤖 Tutor: {response}")

Starting conversation with Alice:

👤 Sujat: Hi! I'm Sujat and I'm learning Python.
🤖 Tutor: Hello Sujat, it's great to meet you. Learning Python can be a fantastic journey, and I'm more than happy to help you along the way. Python is an amazing language with a wide range of applications, from web development to data science and more.

What specific areas of Python are you interested in or struggling with? Are you a complete beginner, or do you have some experience with programming already? Let's chat, and I'll do my best to assist you.

👤 Sujat: What topic should I study after learning variables?
🤖 Tutor: After learning variables, a great topic to study next would be **Data Types** and **Operators**.

Understanding the different data types in Python, such as:

* Integers (int)
* Floating Point Numbers (float)
* Strings (str)
* Boolean (bool)
* Lists (list)
* Tuples (tuple)

will help you to work with different types of data and perform operations on them.

Additionally, learning about 

In [8]:
# ============================================================
# Multiple concurrent sessions (multi-user chatbot!)
# ============================================================
bob_config = {"configurable": {"session_id": "user_bob"}}

# Bob has his own separate conversation
print("Bob's independent session:")
print("=" * 50)

response = chatbot.invoke(
    {"input": "I'm Bob and I'm learning JavaScript, not Python!"},
    config=bob_config
)
print(f"🤖 Tutor: {response}")

# Alice still remembers her context
print("\n\nBack to Alice's session:")
response = chatbot.invoke(
    {"input": "What language am I learning again?"},
    config=config  # Alice's config
)
print(f"🤖 Tutor: {response}")

# Show all sessions
print(f"\n\nActive sessions: {list(session_store.keys())}")
for sid, hist in session_store.items():
    print(f"  {sid}: {len(hist.messages)} messages")

Bob's independent session:
🤖 Tutor: Hello Bob, I'm excited to help you with JavaScript. Don't worry if you see any Python references - we'll focus on JavaScript from now on. What topic in JavaScript are you currently learning or struggling with? I'm here to help you understand it better. Is it variables, functions, loops, or something more advanced like DOM manipulation or asynchronous programming? Let me know, and we'll work through it together.


Back to Alice's session:
🤖 Tutor: You're learning **Python**. We've been discussing various topics related to Python, including variables, data types, and operators.


Active sessions: ['user_alice', 'user_bob']
  user_alice: 10 messages
  user_bob: 2 messages


## 4️⃣ Memory Management — Handling Long Conversations

In [9]:
# ============================================================
# Trimming history — Keep last N messages to avoid token limits
# ============================================================
from langchain_core.messages import trim_messages

# Create a long history
long_history = [SystemMessage(content="You are a helpful assistant.")]
for i in range(10):
    long_history.append(HumanMessage(content=f"This is message {i+1}"))
    long_history.append(AIMessage(content=f"This is response {i+1}"))

print(f"Original history: {len(long_history)} messages")

# Trim to last 6 messages (keep system message)
trimmed = trim_messages(
    long_history,
    max_tokens=500,           # Keep within token budget
    strategy="last",          # Keep the LAST messages
    token_counter=llm,        # Use LLM to count tokens
    include_system=True,      # Always keep system message
    allow_partial=False,
    start_on="human"          # Ensure we start on a human message
)

print(f"Trimmed history: {len(trimmed)} messages")
print("\nTrimmed messages:")
for msg in trimmed:
    print(f"  {type(msg).__name__}: {msg.content[:50]}")

Original history: 21 messages


c:\Users\sujat\projects\AI\.venv\Lib\site-packages\langchain_core\language_models\base.py:354: UserWarning: Using fallback GPT-2 tokenizer for token counting. Token counts may be inaccurate for non-GPT-2 models. For accurate counts, use a model-specific method if available.
  return len(self.get_token_ids(text))


Trimmed history: 21 messages

Trimmed messages:
  SystemMessage: You are a helpful assistant.
  HumanMessage: This is message 1
  AIMessage: This is response 1
  HumanMessage: This is message 2
  AIMessage: This is response 2
  HumanMessage: This is message 3
  AIMessage: This is response 3
  HumanMessage: This is message 4
  AIMessage: This is response 4
  HumanMessage: This is message 5
  AIMessage: This is response 5
  HumanMessage: This is message 6
  AIMessage: This is response 6
  HumanMessage: This is message 7
  AIMessage: This is response 7
  HumanMessage: This is message 8
  AIMessage: This is response 8
  HumanMessage: This is message 9
  AIMessage: This is response 9
  HumanMessage: This is message 10
  AIMessage: This is response 10


In [10]:
# ============================================================
# Summarization Memory — Compress old history into a summary
# ============================================================

class SummarizingChatbot:
    """A chatbot that summarizes old history to save tokens"""
    
    def __init__(self, max_messages: int = 6):
        self.max_messages = max_messages
        self.summary = ""
        self.recent_messages = []
        self.llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)
        
        self.chat_prompt = ChatPromptTemplate.from_messages([
            ("system", "You are a helpful assistant." + 
             (" Previous conversation summary: {summary}" if "{summary}" else "")),
            MessagesPlaceholder("recent_history"),
            ("human", "{input}")
        ])
    
    def _summarize_history(self):
        """Summarize old messages to free up space"""
        messages_to_summarize = self.recent_messages[:-2]  # Keep last 2
        
        summary_chain = (
            ChatPromptTemplate.from_messages([
                ("system", "Summarize this conversation history concisely in 2-3 sentences."),
                MessagesPlaceholder("messages")
            ])
            | self.llm | StrOutputParser()
        )
        
        new_summary = summary_chain.invoke({"messages": messages_to_summarize})
        
        # Combine with old summary
        if self.summary:
            self.summary = f"{self.summary} {new_summary}"
        else:
            self.summary = new_summary
        
        # Keep only last 2 messages
        self.recent_messages = self.recent_messages[-2:]
        print(f"  [Compressed history — new summary: '{new_summary[:60]}...']")
    
    def chat(self, user_input: str) -> str:
        # Check if we need to compress
        if len(self.recent_messages) >= self.max_messages:
            self._summarize_history()
        
        self.recent_messages.append(HumanMessage(content=user_input))
        
        chain = self.chat_prompt | self.llm | StrOutputParser()
        response = chain.invoke({
            "summary": self.summary,
            "recent_history": self.recent_messages[:-1],
            "input": user_input
        })
        
        self.recent_messages.append(AIMessage(content=response))
        return response

# Test it
bot = SummarizingChatbot(max_messages=4)

messages = [
    "My name is Alex and I'm a data scientist.",
    "I work mainly with Python and SQL.",
    "I'm interested in learning about LLMs.",
    "What's the best way to get started?",
    "Can you remember my background when giving advice?"
]

for msg in messages:
    print(f"\n👤 {msg}")
    response = bot.chat(msg)
    print(f"🤖 {response[:150]}..." if len(response) > 150 else f"🤖 {response}")


👤 My name is Alex and I'm a data scientist.
🤖 Nice to meet you, Alex. As a data scientist, you must work with data and analytics on a daily basis. What kind of projects have you been working on la...

👤 I work mainly with Python and SQL.
🤖 Python and SQL are both powerful tools for data science. Python is great for data manipulation, analysis, and machine learning, while SQL is essential...

👤 I'm interested in learning about LLMs.
  [Compressed history — new summary: '...']
🤖 Large Language Models (LLMs) are a fascinating topic in the field of natural language processing (NLP). LLMs are a type of artificial intelligence (AI...

👤 What's the best way to get started?
  [Compressed history — new summary: '...']
🤖 Getting started with Large Language Models (LLMs) can be exciting, but it can also be overwhelming due to the vast amount of information available. He...

👤 Can you remember my background when giving advice?
  [Compressed history — new summary: '...']
🤖 I don't have personal me

## 5️⃣ Persistent Memory with SQLite

In [12]:
# pip install langchain-community
from langchain_community.chat_message_histories import SQLChatMessageHistory

# ============================================================
# SQLite-backed persistent history
# Survives application restarts!
# ============================================================
history = SQLChatMessageHistory(
    session_id="persistent_user_001",
    connection_string="sqlite:///chat_history.db"  # Creates local DB file
)

# Add some messages
history.add_user_message("Remember this: my API key is stored in env vars.")
history.add_ai_message("Got it! I'll remember that about your setup.")

# Retrieve messages (these persist across Python sessions!)
print("Messages in SQLite DB:")
for msg in history.messages:
    print(f"  {type(msg).__name__}: {msg.content}")

# Clean up
import os
if os.path.exists("chat_history.db"):
    os.remove("chat_history.db")
    print("\n(Demo DB cleaned up)")

Messages in SQLite DB:
  HumanMessage: Remember this: my API key is stored in env vars.
  AIMessage: Got it! I'll remember that about your setup.
  HumanMessage: Remember this: my API key is stored in env vars.
  AIMessage: Got it! I'll remember that about your setup.


PermissionError: [WinError 32] The process cannot access the file because it is being used by another process: 'chat_history.db'

## ✅ Module 05 Summary

You've learned:
- ✅ Why LLMs need explicit memory management
- ✅ Manual message list approach (simple)
- ✅ `InMemoryChatMessageHistory` for structured storage
- ✅ `RunnableWithMessageHistory` for production chatbots
- ✅ Multi-user session management
- ✅ `trim_messages()` for token limit management
- ✅ Summarization memory pattern
- ✅ SQLite persistent memory

### 🚀 Next: [Module 06 — Document Loaders & Text Splitters](06_Document_Loaders.ipynb)